# Deliverable 3 — AA Data EDA

**Sahith Yamani** · `team-02/sahith-yamani`  
**My focus:** Leakage — which columns can we use when making a prediction?

This review draft uses [Aidan Chance's target/outlier work](https://github.com/TAMU-Aggie-Data-Science-Club/fall2026-american-airlines/blob/team-02-aidan-chance/team-02-modeling/notebooks/eda_aidan-chance.ipynb) and [Akshith's EDA and time-axis work](https://github.com/TAMU-Aggie-Data-Science-Club/fall2026-american-airlines/blob/team-02/akshithreddy-sheri/team-02-modeling/notebooks/eda_akshithreddy-sheri.ipynb). Team findings below are attributed to them; I have not rerun the AA analysis in this draft.

**NDA:** Keep the files in `data/raw/`. Do not share values, output tables, charts, or screenshots. Conclusions stay in words only. Clear all outputs before committing. The code does not save data or plots.


## Task and validation

Following Akshith's description, the task is regression: predict `OD_PAX`, the passenger measure for an origin–destination route and departure month. Route codes are categories; `DEP_YEAR` and `DEP_MONTH` identify the departure period. Repeated route-month rows must be checked before assuming the key is unique.

The code follows his split: reserve the latest twelve observed departure months and use earlier months for target-based EDA. An explicit `EXPLORATION_BEFORE` can override that boundary if the team changes it. Keep each departure month together in future validation folds.

**Still open:** when the forecast is issued and when historical outcomes become available. I therefore leave lagged passenger features conditional and do not build them here. A chronological split alone does not make every historical feature safe.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

TARGET_COLUMN = "OD_PAX"
TEST_MONTHS = 12  # Match Akshith's most recent observed-month holdout.
EXPLORATION_BEFORE = None  # Optional team-agreed first held-out month.


In [ ]:
# Support launching Jupyter from either the repository or its notebook folder.
repo_root = next(
    (
        folder
        for folder in [Path.cwd(), *Path.cwd().parents]
        if (folder / "AGENTS.md").is_file()
        and (folder / "DATA.md").is_file()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from inside the project repository.")

data_dir = repo_root / "data" / "raw"
pax = pd.read_csv(data_dir / "AA_PAX_FORECASTING.csv")
airports = pd.read_csv(data_dir / "AA_AIRPORT_INFO.csv")

required_pax_columns = {
    "DEP_MONTH", "DEP_YEAR", "ORG_AIRPRT", "DES_AIRPRT",
    "OD_PAX", "OD_REV_USD", "SF_LOAD_TMS",
}
required_airport_columns = {"AIRPRT_CD", "CNTRY_CD", "STATE_PROVNC_CD"}
if not required_pax_columns.issubset(pax.columns):
    raise ValueError("Passenger schema differs from the discussed schema; review it locally.")
if not required_airport_columns.issubset(airports.columns):
    raise ValueError("Airport schema differs from the discussed schema; review it locally.")

print("Both files loaded locally. No sample records are displayed.")


## 1. Shape and types

The passenger table holds route/departure-period fields, passenger and revenue measures, a system load timestamp, and an export row counter. The airport file is a separate reference table keyed by airport code. This follows Akshith's field descriptions; the checks below verify the local schema without printing sample rows.

| Fields | Interpretation and handling |
|---|---|
| `DEP_MONTH`, `DEP_YEAR` | Departure calendar fields; validate calendar values before building a period. |
| `ORG_AIRPRT`, `DES_AIRPRT` | Airport identifiers; categorical even if stored as numbers. |
| `OD_PAX` | Passenger target; floating storage alone is no reason to round it. |
| `OD_REV_USD` | Same-period realized revenue according to Akshith; exclude from predictors. |
| `SF_LOAD_TMS` | System load metadata, separate from the departure date and source availability. |
| `Unnamed: 0` | Export index identified by both teammates; drop from model inputs. |
| Airport attributes | Check code definitions, coordinate directions, units, and historical validity. |


In [ ]:
for label, frame in [("Passenger", pax), ("Airport", airports)]:
    print(label)
    print("Shape:", frame.shape)
    frame.info(verbose=True, show_counts=True)
    display(pd.DataFrame({"stored_type": frame.dtypes.astype(str)}))

index_candidates = [
    column for column in pax.columns
    if str(column).strip().lower().startswith("unnamed:")
    or str(column).strip().lower() == "index"
]
print("Possible exported-index fields to review locally:", index_candidates)


In [ ]:
# Validate possible date conversions without modifying either loaded frame.
parsed_load_time = pd.to_datetime(pax["SF_LOAD_TMS"], errors="coerce")
load_time_parse_failures = (
    pax["SF_LOAD_TMS"].notna() & parsed_load_time.isna()
)

departure_year = pd.to_numeric(pax["DEP_YEAR"], errors="coerce")
departure_month = pd.to_numeric(pax["DEP_MONTH"], errors="coerce")
valid_period_fields = (
    departure_year.notna()
    & departure_month.notna()
    & departure_year.mod(1).eq(0)
    & departure_month.mod(1).eq(0)
    & departure_month.between(1, 12)
)
departure_period = pd.to_datetime(
    {
        "year": departure_year.where(valid_period_fields),
        "month": departure_month.where(valid_period_fields),
        "day": 1,
    },
    errors="coerce",
)

display(pd.Series({
    "nonmissing_load_values_that_failed_parsing": load_time_parse_failures.sum(),
    "records_without_a_valid_departure_period": departure_period.isna().sum(),
}))
# Parsing does not establish the timestamp's business meaning or timezone.


**Conclusion:** Use the field's meaning rather than its stored type. Keep airport codes categorical and build the time axis from the departure year/month. The load timestamp is not a substitute for the departure period or proof that an outcome was available earlier. Resolve invalid dates and unexplained duplicate keys before constructing features.


## 2. Missingness

Both teammates report no pandas-recognized nulls in the passenger table. Akshith reports gaps in airport state/province codes and an association with country. Recheck these locally rather than treating either observation as a rule for every future file. Null checks also miss absent route-month records and undocumented placeholder codes.


In [ ]:
for label, frame in [("Passenger", pax), ("Airport", airports)]:
    missing_summary = pd.DataFrame({
        "missing_count": frame.isna().sum(),
        "missing_percent": frame.isna().mean().mul(100),
    }).sort_values("missing_count", ascending=False)
    print(label)
    display(missing_summary)


In [ ]:
state_missing_by_country = (
    airports.assign(state_missing=airports["STATE_PROVNC_CD"].isna())
    .groupby("CNTRY_CD", dropna=False)["state_missing"]
    .agg(
        airport_records="size",
        missing_records="sum",
        missing_share="mean",
    )
    .sort_values("missing_share", ascending=False)
)
display(state_missing_by_country)

if state_missing_by_country["missing_share"].nunique() > 1:
    print("Observed missingness rates vary between country groups.")
else:
    print("Observed missingness rates are the same across country groups.")


**Conclusion:** The team findings do not call for passenger-table imputation. Airport state/province gaps need separate handling: keep those airport records, check the country pattern, and use an explicit missing category if the field enters the model. Akshith proposes `INTL`; I would use that label only where the documented country rule supports it, because an unknown state is not automatically an international airport.

Do not fill a missing route-month with zero until a PM confirms whether it means no passengers or no record. Any learned filling rule belongs inside the training pipeline.


## 3. Distributions and outliers

Aidan and Akshith both describe a right-skewed passenger target with large values that may represent busy routes. I follow their keep-unless-invalid policy. These exploration-only checks compare the raw and log views, flag unusual values, and inspect physical airport measurements. Revenue is inspected as a suspect field, not approved as a predictor.


In [ ]:
if TARGET_COLUMN not in pax or not pd.api.types.is_numeric_dtype(pax[TARGET_COLUMN]):
    raise ValueError("Confirm the numeric target and validate its type before plotting.")
if departure_period.isna().any():
    raise ValueError("Resolve invalid or missing departure periods before selecting exploration records.")

observed_periods = pd.DatetimeIndex(departure_period.unique()).sort_values()
if EXPLORATION_BEFORE is None:
    if not isinstance(TEST_MONTHS, int) or TEST_MONTHS < 1:
        raise ValueError("TEST_MONTHS must be a positive integer.")
    if len(observed_periods) <= TEST_MONTHS:
        raise ValueError("Not enough departure months for the lead's holdout; agree a shorter holdout.")
    exploration_cutoff = observed_periods[-TEST_MONTHS]
else:
    exploration_cutoff = pd.Timestamp(EXPLORATION_BEFORE)
    if pd.isna(exploration_cutoff) or exploration_cutoff.tzinfo is not None:
        raise ValueError("Use a valid timezone-free departure month for the exploration boundary.")
    if exploration_cutoff != exploration_cutoff.to_period("M").to_timestamp():
        raise ValueError("Use the first day of the first held-out departure month.")

exploration_mask = departure_period.lt(exploration_cutoff)
if not exploration_mask.any() or exploration_mask.all():
    raise ValueError("The boundary must leave both exploration and held-out records.")

eda_pax = pax.loc[exploration_mask].copy()
eda_period = departure_period.loc[exploration_mask]
print("Earlier months selected for exploration. Held-out target values are not used below.")


In [ ]:
# Revenue is inspected as a suspect field, not approved as an input feature.
outcome_measures = list(dict.fromkeys([TARGET_COLUMN, "OD_REV_USD"]))
for column in outcome_measures:
    if not pd.api.types.is_numeric_dtype(eda_pax[column]):
        print("Validate this measure's numeric type locally before plotting:", column)
        continue
    values = eda_pax[column].dropna()
    if values.empty:
        print("No nonmissing exploration values available for:", column)
        continue
    if not np.isfinite(values).all():
        raise ValueError("Resolve non-finite exploration values before plotting.")
    print(column)
    display(values.describe())
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    values.plot.hist(bins=30, ax=axes[0], title=f"{column}: distribution")
    values.plot.box(ax=axes[1], title=f"{column}: spread")
    plt.tight_layout()
    plt.show()

target_values = eda_pax[TARGET_COLUMN].dropna()
if not target_values.empty and target_values.ge(0).all():
    log_target = np.log1p(target_values)
    display(pd.Series({"raw_skew": target_values.skew(), "log1p_skew": log_target.skew()}))
    log_target.plot.hist(bins=30, title="Exploration-only log1p target (diagnostic)")
    plt.show()
else:
    print("Log view skipped: check missing or negative passenger values; no clipping was applied.")


In [ ]:
# A boxplot-style screen flags unusual values; it does not identify errors.
outlier_review = []
for column in outcome_measures:
    if not pd.api.types.is_numeric_dtype(eda_pax[column]):
        continue
    values = eda_pax[column].dropna()
    if values.empty:
        continue
    lower_quartile, upper_quartile = values.quantile([0.25, 0.75])
    interquartile_range = upper_quartile - lower_quartile
    if interquartile_range > 0:
        flags = (
            values.lt(lower_quartile - 1.5 * interquartile_range)
            | values.gt(upper_quartile + 1.5 * interquartile_range)
        )
        flagged_count = flags.sum()
        flagged_share = flags.mean()
    else:
        flagged_count = pd.NA
        flagged_share = pd.NA
    outlier_review.append({
        "measure": column,
        "iqr_screen_available": interquartile_range > 0,
        "flagged_count": flagged_count,
        "flagged_share": flagged_share,
        "negative_count": values.lt(0).sum(),
        "zero_count": values.eq(0).sum(),
    })
display(pd.DataFrame(outlier_review))


In [ ]:
# Candidate physical measurements are described separately; no join is assumed.
for column in ["LNGST_RUNWAY_FT_QTY", "ELEVATN_FT_QTY"]:
    if column not in airports or not pd.api.types.is_numeric_dtype(airports[column]):
        continue
    values = airports[column].dropna()
    if values.empty:
        continue
    print(column)
    display(values.describe())
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    values.plot.hist(bins=30, ax=axes[0], title=f"{column}: distribution")
    values.plot.box(ax=axes[1], title=f"{column}: spread")
    plt.tight_layout()
    plt.show()


**Conclusion:** Keep plausible high-demand routes. An IQR flag identifies unusual values, not errors; negative or otherwise implausible values need source clarification rather than silent clipping. This notebook does not drop, cap, or transform the training target.

There is a real choice to carry into model comparison: Aidan recommends squared error on `log1p(OD_PAX)`, while Akshith recommends a raw-target count objective such as Poisson or Tweedie. The log view is useful evidence about shape, but a more symmetric histogram does not settle the loss. Directly reversing a log prediction does not generally recover the conditional mean on the passenger scale. Compare predictions on that scale under the same time split before choosing a loss.


## 4. Leakage risk

Ask of every column: **Would we have this value when making the prediction?**

- **`OD_PAX`: target only.** Never include the answer, its log transform, or a same-row derivative among the predictors.
- **`OD_REV_USD`: leak / drop.** Akshith identifies it as realized revenue for the same route-month; Aidan also flags it. It is outcome information for a forecast made before that month. Its correlation with passengers can support an investigation, but timing is the reason to exclude it. Revenue per passenger and similar derivatives inherit this problem.
- **`Unnamed: 0`: ID / drop.** Both teammates identify an export counter. An arbitrary row number can encode ordering without representing demand.
- **`SF_LOAD_TMS`: metadata / drop from inputs.** Keep it outside the model for audit purposes. A system load time is not evidence that the underlying value was known before a forecast; delayed loads and revisions matter.
- **Departure fields and route codes: keep as candidates.** The requested route and forecast month can be known in advance. They still require suitable encoding and time-based evaluation.
- **Airport details: conditional.** A valid code match is insufficient if the reference snapshot contains later runway, location, or administrative information.

**Conclusion:** Exclude the outcome and its same-period proxies. Time-based validation, training-only preprocessing, and historically valid reference joins address additional leakage that a column blacklist cannot catch.


## 5. Relationships and model choice

Both teammates point toward boosted trees for route/category differences and nonlinear calendar effects. Akshith also reports recurring calendar patterns and changes over time. The cells below let me check these qualitatively on the earlier months without copying their results into this notebook.

Airport codes are labels. A tree model still needs an appropriate encoder or configured categorical support; numerical code order does not represent airport similarity. Learn encodings from training data only.


In [ ]:
category_summary = []
for column in ["ORG_AIRPRT", "DES_AIRPRT"]:
    category_summary.append({
        "feature": column,
        "distinct_categories": eda_pax[column].nunique(dropna=False),
    })
    grouped_target = (
        eda_pax.groupby(column, dropna=False)[TARGET_COLUMN]
        .agg(records="size", target_mean="mean", target_median="median")
    )
    print(column)
    # Aggregate groups only; no raw passenger records are displayed.
    display(grouped_target.nlargest(12, "records"))
display(pd.DataFrame(category_summary))

print("Distinct observed origin/destination pairs:")
print(eda_pax.groupby(["ORG_AIRPRT", "DES_AIRPRT"], dropna=False).ngroups)


In [ ]:
# Association with suspect revenue is diagnostic, not permission to use it.
if TARGET_COLUMN != "OD_REV_USD" and pd.api.types.is_numeric_dtype(eda_pax["OD_REV_USD"]):
    related_measures = eda_pax[[TARGET_COLUMN, "OD_REV_USD"]].dropna()
    display(related_measures.corr())
    if not related_measures.empty:
        related_measures.plot.hexbin(
            x="OD_REV_USD", y=TARGET_COLUMN, gridsize=30, mincnt=1,
            title="Exploration-only association; revenue availability unconfirmed",
        )
        plt.show()
else:
    print("Review a distinct, confirmed numerical relationship if applicable.")


In [ ]:
calendar_view = eda_pax.assign(
    departure_period_for_eda=eda_period,
    departure_month_for_eda=departure_month.loc[eda_pax.index],
)
route_month_columns = ["ORG_AIRPRT", "DES_AIRPRT", "departure_period_for_eda"]
repeated_route_periods = calendar_view.duplicated(subset=route_month_columns)
route_month_target_counts = (
    calendar_view.groupby(route_month_columns, dropna=False)[TARGET_COLUMN]
    .nunique(dropna=False)
)
route_count = calendar_view.groupby(["ORG_AIRPRT", "DES_AIRPRT"], dropna=False).ngroups
exploration_months = pd.period_range(eda_period.min(), eda_period.max(), freq="M")
observed_route_months = len(calendar_view.drop_duplicates(route_month_columns))
display(pd.Series({
    "distinct_departure_periods": eda_period.nunique(),
    "repeated_route_period_records": repeated_route_periods.sum(),
    "route_period_keys_with_different_target_values": route_month_target_counts.gt(1).sum(),
    "unobserved_route_month_combinations_in_exploration_range": (
        route_count * len(exploration_months) - observed_route_months
    ),
}))
# A missing combination may mean a new/closed route or no record, not zero passengers.
# Repeated keys may be snapshots or distinct subgroups; do not sum/deduplicate blindly.

period_summary = (
    calendar_view.groupby("departure_period_for_eda")[TARGET_COLUMN]
    .agg(records="size", target_mean="mean", target_median="median")
    .sort_index()
)
display(period_summary)
if not period_summary.empty:
    period_summary["target_mean"].plot(
        marker="o", title="Exploration-only mean across records by departure period"
    )
    plt.ylabel("Mean of the target measure")
    plt.show()

monthly_summary = (
    calendar_view.groupby("departure_month_for_eda")[TARGET_COLUMN]
    .agg(records="size", target_mean="mean", target_median="median")
    .sort_index()
)
display(monthly_summary)
# Changes in route composition can move these averages; they do not prove seasonality.


In [ ]:
# Audit join keys without joining or exposing individual records.
airport_keys = airports["AIRPRT_CD"].dropna()
display(pd.Series({
    "airport_records_with_missing_key": airports["AIRPRT_CD"].isna().sum(),
    "airport_records_with_repeated_key": airport_keys.duplicated(keep=False).sum(),
    "exploration_origin_match_share": eda_pax["ORG_AIRPRT"].isin(airport_keys).mean(),
    "exploration_destination_match_share": eda_pax["DES_AIRPRT"].isin(airport_keys).mean(),
}))
# Use separate origin/destination reference joins, each validated as many-to-one.
# Resolve duplicate keys and historic validity before joining; retain unmatched rows for review.
# Code-type mismatches require checking the documented code system, not blind coercion.


**Conclusion:** The route categories and calendar effects described by Aidan and Akshith support evaluating a model that can learn interactions between route and month. Category cardinality affects representation; rare or unseen routes need explicit handling. Revenue association adds no permission to use revenue.

Use earlier departure periods for training and recent periods for testing, following the lead's rule. For rolling validation, split distinct months first and map each month to all its rows; a row-level time split could otherwise cut through the same month. Changes in route mix can move overall averages, so they do not by themselves prove a seasonal pattern for a fixed route. Repeated route-months require clarification before aggregation.


## 6. My deep-dive — leakage hand-off

My hand-off separates fields we can use from information that would reveal the answer. The audit below covers every loaded column. Unexpected fields remain excluded pending review; it does not build the team's final feature set.

The column decisions also require these process rules:

- Fit imputation, encoding, scaling, and feature selection on the training months only. Target encodings for training rows must avoid using that row's own outcome and later outcomes.
- Never compute route target means on the full dataset. Same-month totals, target-derived fares, and a log of the current target are still outcome information.
- A lag needs an exact earlier calendar month, the same route, and an outcome available before the forecast was issued. A row shift across a gap can select the wrong month.
- For one forecast covering the whole holdout, freeze actual history at the forecast origin; later held-out outcomes cannot enter lag features. With monthly rolling forecasts, new actuals can enter only after they become available. The team still needs to specify this timing and any reporting delay.
- Do not silently sum or drop repeated route-months. They may be revisions or subgroups, and a later revision may reveal information unavailable at the original forecast.
- Airport joins must preserve passenger rows, use a unique reference key for each airport role, and respect historical validity. Fit any learned reference-based transform on training data only.


In [ ]:
passenger_proposals = {
    "DEP_MONTH": ("keep / engineer", "Requested departure month is known; categorical/cyclic handling belongs to the feature owner."),
    "DEP_YEAR": ("keep", "Requested departure year is known; preserve chronological validation."),
    "ORG_AIRPRT": ("keep / categorical", "Requested origin is known; an airport code is a category, not a row counter."),
    "DES_AIRPRT": ("keep / categorical", "Requested destination is known; handle unseen airports using training-fitted rules."),
    "OD_PAX": ("target only", "Passenger outcome, following the lead's task definition; never an input to its own prediction."),
    "OD_REV_USD": ("leak / drop", "The lead identifies same-route-month realized revenue, unavailable before that month."),
    "SF_LOAD_TMS": ("metadata / drop", "System load timestamp; retain outside inputs to investigate delays and revisions."),
}
airport_proposals = {
    "AIRPRT_CD": ("id / join key", "Verify code system and unique reference key; use separately for origin and destination."),
    "AIRPRT_NM": ("drop from initial inputs", "Keep for interpretation; the name duplicates the airport identity and adds no verified available signal."),
    "CITY_METRO_IATA_CD": ("conditional keep / categorical", "Location context only after confirming definitions, key matching, and historical validity."),
    "STATE_PROVNC_CD": ("conditional keep / categorical", "Requires historical validity and documented missing-category handling; do not assume every gap means international."),
    "CNTRY_CD": ("conditional keep / categorical", "Geographical context if the reference was valid at the forecast point."),
    "WRLD_AREA_DOT_CD": ("withhold pending definition", "Do not infer a continuous measurement or category meaning from the stored code."),
    "SF_LOAD_TMS": ("metadata / drop", "Reference-load metadata is not a predictor or proof of historical attribute availability."),
    "LNGST_RUNWAY_FT_QTY": ("conditional keep", "Physical attribute only if its units and historical measurement/snapshot date are established."),
    "ELEVATN_FT_QTY": ("conditional keep", "Physical attribute only after validating units, missingness, and historical reference validity."),
}
for column in [
    "LAT_DEGREE_QTY", "LAT_MINUTE_QTY", "LAT_SECOND_QTY", "LAT_HEMSPHR_CD",
    "LNGTD_DEGREE_QTY", "LNGTD_MINUTE_QTY", "LNGTD_SECOND_QTY", "LNGTD_HEMSPHR_CD",
]:
    airport_proposals[column] = (
        "conditional engineer",
        "Verify coordinate components, direction codes, and historical validity before combining; no raw-code arithmetic.",
    )

audit_rows = []
for source_name, frame, proposals in [
    ("Passenger", pax, passenger_proposals),
    ("Airport", airports, airport_proposals),
]:
    for column in frame.columns:
        decision, reason = proposals.get(
            column,
            ("withhold pending definition", "Exclude until the lead confirms meaning and forecast-time availability."),
        )
        if source_name == "Passenger" and column == TARGET_COLUMN:
            decision = "target only / drop from inputs"
            reason = "Retain the outcome as the training label, separate from predictors."
        elif source_name == "Passenger" and str(column).strip().lower() == "unnamed: 0":
            decision = "id / drop"
            reason = "Export-counter field identified in the team review; not a demand measurement."
        audit_rows.append({
            "Column / feature": f"{source_name}: {column}",
            "Decision (keep / drop / engineer / leak / id)": decision,
            "Reason": reason,
        })

leakage_audit = pd.DataFrame(audit_rows)
display(leakage_audit)
# Decisions only: this cell neither changes the source frames nor builds a feature matrix.


### Hand-off to Akshith

| Column / feature | Decision | Reason |
|---|---|---|
| `OD_PAX` and a transformation of the current target | Target only; drop from inputs | The outcome and its derivatives reveal the answer. |
| `OD_REV_USD` and same-period revenue/passenger ratios | Leak / drop | Same-period realized revenue is known too late for the forecast described by the lead. |
| `Unnamed: 0` / export counter | ID / drop | Export ordering is not a demand measurement. |
| Passenger and airport `SF_LOAD_TMS` | Drop from inputs; retain for auditing | Processing metadata; inspect delays and revisions separately. |
| `ORG_AIRPRT`, `DES_AIRPRT` | Keep as categories | Requested route is known; fit encoding on training months. |
| `DEP_YEAR`, `DEP_MONTH` | Keep / engineer | Forecast period is known; validate on later months. |
| `AIRPRT_CD` | ID / join key | Check key uniqueness, code matching, and historical reference validity. |
| `AIRPRT_NM` | Drop from initial inputs | Keep as a human-readable label rather than duplicate identity. |
| City, state/province, and country codes | Conditional keep | Historical reference and missing-code handling must be valid. |
| `WRLD_AREA_DOT_CD` | Withhold pending definition | The stored code alone does not establish its meaning. |
| Runway length and elevation | Conditional keep | Verify units, gaps, and historical attribute availability. |
| Coordinate components and hemisphere codes | Conditional engineer | Verify definitions before combining; use a historically valid reference. |
| `pax_same_month_last_year`, `route_seen_last_year` | Conditional historical features | Exact calendar match and available-at-forecast history are required; a current extract can include later revisions. |
| Full-dataset route means, same-month target aggregates | Leak / drop | They can include the current answer or future/held-out outcomes. |
| Unrecognized columns | Withhold pending review | No field enters inputs solely because it is present in the file. |

**Hand-off:** Exclude current passenger outcomes, realized same-period revenue and their derivatives, export counters, and load timestamps from the inputs. Keep the requested route and departure period. Use airport attributes only after a validated historical join. Learn preprocessing from training months and keep all rows of a departure month together in validation.

**Open for the lead/PM:** confirm the forecast issue date and reporting delay before approving lags; establish historical airport-reference availability; clarify repeated route-month records and whether an absent record means zero demand. These questions do not change the direct outcome/revenue exclusions.


## 7. My model-family view

I recommend **gradient-boosted regression** as the family to evaluate, consistent with both teammates' EDA. This is a family recommendation, not a claim that a model has won a comparison.

| Reason | Evidence and consequence |
|---|---|
| Route/category differences | Aidan and Akshith describe many route/airport categories. Use an encoder or supported categorical configuration; do not treat code order as a distance. |
| Calendar effects and interactions | Akshith describes recurring month effects and changes over time. Boosted trees can model route-by-month interactions; evaluate them on later periods. |
| Uneven passenger scale | Both describe a skewed target and plausible busy-route extremes. Retain valid observations and compare the proposed losses on the passenger scale. |

Keep **Ridge** as a simpler comparison and a seasonal historical baseline only where its inputs meet the forecast-time rules. The loss remains a documented team choice: Aidan's log-target squared error and Akshith's raw-target count objective should be compared rather than treated as equivalent.

The main limitation is availability: a strong score cannot rescue a feature known only after the forecast. Airport snapshots and lagged outcomes still need timing checks. Boosted trees also need careful handling of new airports/routes and do not reliably extrapolate a future trend beyond their training history. Use the lead's chronological holdout, with month-based rolling folds for tuning. This notebook trains no models and reports no new scores.


## References and review status

Team evidence comes from the linked Aidan Chance and Akshith notebooks. Their qualitative findings are attributed above; my code is ready for local verification and contains no copied numerical results.

Method references:

- [Leakage and training-only preprocessing](https://scikit-learn.org/stable/common_pitfalls.html)
- [Missing values](https://scikit-learn.org/stable/modules/impute.html)
- [Time-based validation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html)
- [Linear models](https://scikit-learn.org/stable/modules/linear_model.html)
- [Tree ensembles](https://scikit-learn.org/stable/modules/ensemble.html)

**Review status:** The written leakage hand-off is filled in. AA results have not been regenerated for this edit; verify the diagnostics locally before submission. The genuine timing/reference questions are listed in the hand-off.

Keep these changes local for Sahith's review. Before any later commit, run locally as permitted, check the conclusions, then **Restart & Clear Output** and save. Akshith reviews my work; I review Aidan. A later PR targets `team-02/main`, and the lead handles merging.
